# Geo-Nexus v3.2 — Domain-Adaptive Pretraining (DAPT)
### Stage 4: DeCUR Cross-Modal Continued Pretraining on Maharashtra (Optical Sentinel-2 + SAR Sentinel-1)

**Pretrained Foundation Encoders:**
- Optical branch: SSL4EO-S12 ResNet-18 MoCo-v2 (13 bands)
- SAR branch: BigEarthNet v2.0 Sentinel-1 ResNet-18 (2 bands, dual-pol VV/VH)

Mounted datasets on Kaggle:
- `/kaggle/input/geonexus-mh-v3/`: Bi-temporal patches from Pune and Satara
- `/kaggle/input/ssl4eo-weights/`: Pretrained foundation weights

In [ ]:
# ============ CELL 1: setup + resume ============
import os, time, json, torch, numpy as np
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler

CKPT  = '/kaggle/working/dapt_last.pth'
START, WALL = time.time(), 11 * 3600      # Kaggle kills at 12 h with NO warning. Stop at 11 h so checkpoint is valid.
def out_of_time(): return (time.time() - START) > WALL

print('GPUs available:', torch.cuda.device_count())
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = {
  'epochs': 100, 'batch_size': 96,
  'lr_backbone': 3e-5, 'lr_head': 3e-4,    # 10x gap -- do not equalise
  'warmup_epochs': 5,
  'decur': {'proj_dim': 2048, 'common_ratio': 0.875, 'lambd': 0.0051},
  'temporal': {'tau': 0.1, 'gamma': 0.3},
  'loss_weights': {'decur': 1.0, 'temporal': 0.5},
  'occlusion': {'p_apply': 0.5, 'max_cloud_frac': 0.7},
}

In [ ]:
# ============ CELL 2: build + load pretrained weights ============
from models.stem import verify_stem_surgery
from ssl.pretrain_dapt import DAPTModel, build_optimizer, dapt_epoch
from ssl.decur import DeCUR
from models.dataset import MHPatches

# Pretrained foundation initializations
# Optical: SSL4EO-S12 ResNet-18 MoCo-v2 (13-channel L1C)
s2_ckpt = torch.load('/kaggle/input/ssl4eo-weights/resnet18_s2c_moco.pth', map_location='cpu')

# SAR: BigEarthNet v2.0 Sentinel-1 ResNet-18 (2-channel dual-pol GRD trunk)
# v3.2 Amendment: explicitly loads resnet18_s1_bigearthnet.pth
s1_ckpt = torch.load('/kaggle/input/ssl4eo-weights/resnet18_s1_bigearthnet.pth', map_location='cpu')

# Stem Surgery Gate on Optical S2 conv1 weight
verify_stem_surgery(s2_ckpt['conv1.weight'])     # MUST pass, cosine > 0.98

model = DAPTModel(**{'proj_dim': cfg['decur']['proj_dim']}).to(dev)
model.encoder.load_pretrained(s2_ckpt, s1_ckpt)

ds = MHPatches(zones=('pune', 'satara'), split='train')
dl = DataLoader(ds, batch_size=cfg['batch_size'], shuffle=True,
                num_workers=2, pin_memory=True, drop_last=True)   # drop_last=True required for BatchNorm1d in projector
print(f'DAPT corpus: {len(ds)} bi-temporal pairs')

decur  = DeCUR(cfg['decur']['proj_dim'], cfg['decur']['common_ratio'],
               cfg['decur']['lambd']).to(dev)
opt    = build_optimizer(model, cfg)
scaler = GradScaler()
sched  = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[cfg['lr_backbone'], cfg['lr_backbone'], cfg['lr_head']],
    total_steps=cfg['epochs'] * len(dl),
    pct_start=cfg['warmup_epochs'] / cfg['epochs'])

In [ ]:
# ============ CELL 3: train ============
start_ep, hist = 0, []
if os.path.exists(CKPT):
    s = torch.load(CKPT, map_location='cpu')
    model.load_state_dict(s['model'])
    opt.load_state_dict(s['opt'])
    scaler.load_state_dict(s['scaler'])
    sched.load_state_dict(s['sched'])
    start_ep, hist = s['epoch'] + 1, s['hist']
    print(f'Resumed training at epoch {start_ep}')

for ep in range(start_ep, cfg['epochs']):
    if ep == 10:                          # restore weight decay on the derived stem
        opt.param_groups[1]['weight_decay'] = 0.05
        print('Derived-stem weight decay re-enabled')

    m = dapt_epoch(model, dl, opt, scaler, decur, cfg, dev)
    sched.step()
    hist.append({'epoch': ep, **m})
    print(f"ep {ep:3d}  total {m['total']:8.1f}  com {m['com']:7.1f}  "\
          f"uni {m['uni']:7.1f}  opt {m['opt']:7.1f}  sar {m['sar']:7.1f}  "\
          f"temp {m['temporal']:.3f}")

    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'scaler': scaler.state_dict(), 'sched': sched.state_dict(),
                'epoch': ep, 'hist': hist, 'cfg': cfg}, CKPT)

    if out_of_time():
        print('Wall-clock guard hit -- checkpoint saved, rerun notebook to resume')
        break

# Save ENCODERS ONLY. The projectors are discarded after DAPT.
torch.save({'encoder': model.encoder.state_dict(), 'cfg': cfg, 'hist': hist},
           '/kaggle/working/dapt_encoders.pth')
json.dump(hist, open('/kaggle/working/dapt_history.json', 'w'), indent=2)
print('DAPT completed successfully. Encoders saved to /kaggle/working/dapt_encoders.pth')

In [ ]:
# ============ CELL 4: post-DAPT t-SNE validation ============
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

model.eval()
Z = []
with torch.no_grad():
    for i, b in zip(range(6), DataLoader(ds, batch_size=64)):
        zo, zs = model.project(b['x'][:, 0].to(dev).float())
        Z.append((zo.cpu().numpy(), zs.cpu().numpy()))
zo = np.concatenate([a for a, _ in Z])
zs = np.concatenate([b for _, b in Z])
emb = TSNE(n_components=2, random_state=0).fit_transform(np.concatenate([zo, zs]))

plt.figure(figsize=(8, 6))
plt.scatter(*emb[:len(zo)].T, s=6, label='optical', alpha=0.7)
plt.scatter(*emb[len(zo):].T, s=6, label='SAR', alpha=0.7)
plt.title('t-SNE of DAPT Learned Representations (Optical vs SAR)')
plt.xlabel('t-SNE dim 1')
plt.ylabel('t-SNE dim 2')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('/kaggle/working/dapt_tsne.png', dpi=300)
plt.show()